# BEV Adoption Rate — Model globalny, prognoza dla Polski
**Dataset:** Global Electric Vehicle Dataset 2023 (Kaggle)  
**Cel:** Prognoza adopcji BEV w Polsce do 2030 na podstawie danych globalnych

## 1. Wczytanie danych

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import kagglehub
from kagglehub import KaggleDatasetAdapter

%matplotlib inline

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "padmapiyush/global-electric-vehicle-dataset-2023",
    ""
)

print("Rozmiar danych:", df.shape)
print("Kolumny:", list(df.columns))
print("Parametry:", sorted(df["parameter"].unique().tolist()))
df.head()

## 2. Analiza danych (EDA)

In [ ]:
BEV            = 'BEV'
PARAM_SALES    = 'EV sales'
PARAM_STOCK    = 'EV stock'
PARAM_CHARGING = 'EV charging points'

# filtruj BEV i wyodrebnij stacje ladowania (przed filtrem BEV)
df_bev = df[df['powertrain'] == BEV].copy()
df_cs  = df[df['parameter'] == PARAM_CHARGING].copy()

print("Dane BEV:", df_bev.shape)
print("Braki danych:\n", df_bev.isnull().sum())
df_bev.describe().round(2)

In [ ]:
# sprzedaz BEV na swiecie w kolejnych latach
df_sales = df_bev[df_bev['parameter'] == PARAM_SALES]
trend = df_sales.groupby('year')['value'].sum()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trend.index, trend.values, marker='o', color='steelblue')
ax.set_title('Laczna sprzedaz BEV na swiecie')
ax.set_xlabel('Rok')
ax.set_ylabel('Liczba sprzedanych BEV')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

In [ ]:
# top 10 krajow wedlug sprzedazy BEV
top10 = df_sales.groupby('region')['value'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
top10.plot(kind='bar', color='steelblue')
plt.title('Top 10 krajow wedlug sprzedazy BEV')
plt.ylabel('Liczba sprzedanych BEV')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# publiczne stacje ladowania EV na swiecie
cs_trend = df_cs.groupby('year')['value'].sum()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(cs_trend.index, cs_trend.values, marker='o', color='darkorange')
ax.set_title('Laczna liczba stacji ladowania EV na swiecie')
ax.set_xlabel('Rok')
ax.set_ylabel('Liczba stacji ladowania')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

In [ ]:
# korelacja: stacje ladowania vs sprzedaz BEV (per kraj-rok)
bev_agg = df_sales.groupby(['region','year'])['value'].sum().reset_index().rename(columns={'value':'bev_sales'})
cs_agg  = df_cs.groupby(['region','year'])['value'].sum().reset_index().rename(columns={'value':'charging_stations'})
corr_df = bev_agg.merge(cs_agg, on=['region','year'], how='inner')

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(corr_df['charging_stations'], corr_df['bev_sales'], alpha=0.4, color='steelblue', edgecolors='none')
ax.set_title('Korelacja: stacje ladowania vs sprzedaz BEV')
ax.set_xlabel('Liczba stacji ladowania')
ax.set_ylabel('Sprzedaz BEV')
plt.tight_layout()
plt.show()

r = corr_df[['charging_stations','bev_sales']].corr().iloc[0,1]
print('Korelacja Pearsona: %.4f' % r)

In [ ]:
# macierz korelacji
numeric_cols = df_bev.select_dtypes(include='number')
plt.figure(figsize=(8, 6))
sns.heatmap(numeric_cols.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Macierz korelacji')
plt.tight_layout()
plt.show()

## 3. Preprocessing i Feature Engineering

In [ ]:
# tylko dane historyczne (do 2023)
df_clean = df_bev[df_bev['year'] <= 2023].copy()
before = len(df_clean)
df_clean = df_clean.drop_duplicates().dropna(subset=['region','year','value'])
print("Usunieto %d wierszy" % (before - len(df_clean)))

# reshape long -> wide: jedna kolumna na parametr
df_s  = df_clean[df_clean['parameter'] == PARAM_SALES][['region','year','value']].rename(columns={'value':'ev_sales'})
df_st = df_clean[df_clean['parameter'] == PARAM_STOCK][['region','year','value']].rename(columns={'value':'ev_stock'})
df_wide = df_s.merge(df_st, on=['region','year'], how='inner')

# dolacz stacje ladowania (agregat per kraj-rok)
df_cs_hist = (df_cs[df_cs['year'] <= 2023]
              .groupby(['region','year'])['value'].sum()
              .reset_index().rename(columns={'value':'charging_stations'}))
df_wide = df_wide.merge(df_cs_hist, on=['region','year'], how='left')
df_wide = df_wide.sort_values(['region','year'])
df_wide['charging_stations'] = (df_wide.groupby('region')['charging_stations']
                                 .transform(lambda s: s.ffill().bfill()).fillna(0))

# zmienna docelowa: adopcja BEV (%) — przycieta do 100%
df_wide['ev_adoption_rate'] = df_wide['ev_sales'] / df_wide['ev_stock'] * 100
df_wide = df_wide[df_wide['ev_adoption_rate'] <= 100].copy()

# wzrost rok do roku
df_wide['yoy_growth'] = df_wide.groupby('region')['ev_sales'].pct_change() * 100

# zachowaj mapowanie regionow przed enkodowaniem
region_categories = sorted(df_wide['region'].unique())
region_to_code = {r: i for i, r in enumerate(region_categories)}
df_wide['region'] = df_wide['region'].astype('category').cat.codes

df_clean = df_wide.dropna(subset=['ev_adoption_rate','yoy_growth']).copy()
print("Dane po preprocessingu:", df_clean.shape)
print("Cechy:", [c for c in df_clean.columns if c != "ev_adoption_rate"])
df_clean['ev_adoption_rate'].describe().round(2)

## 4 & 5. Model i Optymalizacja Hiperparametrow

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score
import optuna

seed      = 42
test_size = 0.33
target    = 'ev_adoption_rate'

drop_cols = [c for c in [target, 'ev_sales', 'ev_stock'] if c in df_clean.columns]
X = df_clean.drop(columns=drop_cols)
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed)
print("Train:", X_train.shape, "  Test:", X_test.shape)
print("Features:", list(X.columns))

In [ ]:
def objective(trial):
    n_estimators     = trial.suggest_int('n_estimators', 50, 400)
    max_depth        = trial.suggest_int('max_depth', 1, 10)
    learning_rate    = trial.suggest_float('learning_rate', 1e-3, 0.3, log=True)
    subsample        = trial.suggest_float('subsample', 0.6, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0)

    model = XGBRegressor(
        n_estimators=n_estimators, max_depth=max_depth,
        learning_rate=learning_rate, subsample=subsample,
        colsample_bytree=colsample_bytree, random_state=seed, verbosity=0
    )
    return cross_val_score(model, X_train, y_train, cv=3,
                           scoring='neg_root_mean_squared_error', n_jobs=-1).mean()


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

trial = study.best_trial
print('Accuracy: {}'.format(trial.value))
print('Best hyperparameters: {}'.format(trial.params))

model = XGBRegressor(**trial.params, random_state=seed, verbosity=0)
model.fit(X_train, y_train)

In [ ]:
optuna.visualization.plot_optimization_history(study)

In [ ]:
optuna.visualization.plot_param_importances(study)

## 6. Interpretacja Wynikow

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import shap

y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
print('RMSE: %.4f | MAE: %.4f | R2: %.4f' % (rmse, mae, r2))

In [ ]:
# feature importance
fi = pd.Series(model.feature_importances_, index=X_test.columns).sort_values(ascending=False)
fi.plot(kind='barh', title='Feature Importance - XGBoost')
plt.tight_layout()
plt.show()

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

top3 = fi.head(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
PartialDependenceDisplay.from_estimator(model, X_test.astype(float), top3, ax=axes)
plt.suptitle('Partial Dependence Plots')
plt.tight_layout()
plt.show()

## 7. Prognoza adopcji BEV w Polsce do 2030

In [ ]:
poland_code = region_to_code.get('Poland', -1)
poland_rows = df_clean[df_clean['region'] == poland_code].sort_values('year')

last_row = poland_rows.iloc[-1].copy()
avg_yoy  = poland_rows['yoy_growth'].tail(3).mean()

# projekcja stacji ladowania srednim tempem wzrostu z ostatnich 3 lat
cs = poland_rows['charging_stations']
cs_growth = cs.pct_change().dropna()
cs_growth = cs_growth[cs_growth != 0].tail(3)
avg_cs_growth = cs_growth.mean() if not cs_growth.empty else 0.05

future_years = list(range(2024, 2031))
projected_cs, val = [], cs.iloc[-1]
for _ in future_years:
    val = val * (1 + avg_cs_growth)
    projected_cs.append(val)

# buduj wiersze do predykcji
feature_cols = [c for c in df_clean.columns if c != target]
future_rows = []
for year, cs_val in zip(future_years, projected_cs):
    row = last_row[feature_cols].copy()
    row['year'] = year
    row['yoy_growth'] = avg_yoy
    row['charging_stations'] = cs_val
    future_rows.append(row)

X_future = pd.DataFrame(future_rows)
predictions = model.predict(X_future)

for year, val in zip(future_years, predictions):
    print('%d: %.4f%%' % (year, val))

In [ ]:
historical = poland_rows.groupby('year')['ev_adoption_rate'].mean()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(historical.index, historical.values, marker='o', color='steelblue', label='Dane historyczne (Polska)')
ax.plot(future_years, predictions, marker='o', linestyle='--', color='orange', label='Prognoza XGBoost')
ax.axvline(x=2023, color='gray', linestyle=':', label='Granica prognozy')
ax.set_title('Prognoza adopcji BEV w Polsce do 2030')
ax.set_xlabel('Rok')
ax.set_ylabel('Adopcja BEV (%)')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend()
plt.tight_layout()
plt.show()

# projekcja stacji ladowania
hist_cs = poland_rows.groupby('year')['charging_stations'].mean()
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hist_cs.index, hist_cs.values, marker='o', color='darkorange', label='Dane historyczne')
ax.plot(future_years, projected_cs, marker='o', linestyle='--', color='red', label='Projekcja')
ax.axvline(x=2023, color='gray', linestyle=':')
ax.set_title('Projekcja stacji ladowania EV w Polsce do 2030')
ax.set_xlabel('Rok')
ax.set_ylabel('Liczba stacji ladowania')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend()
plt.tight_layout()
plt.show()